# 10 — Whole-file tensor I/O benchmark

Compare four separate frame reads with one whole-file read
on the same 512 real 2013 training files.

Call order alternates by file to reduce warm-cache bias.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.7"
BENCHMARK_FILE_COUNT = 512

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
DRIVE_ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)
RESULT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "io_benchmark_v1"
)
RESULT_PATH = (
    RESULT_DIRECTORY
    / "2013_file_tensor_benchmark.json"
)

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_2013_io_benchmark"
)

for required_path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    DRIVE_ARCHIVE_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required path: {required_path}"
        )

if RESULT_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite {RESULT_PATH}"
    )

print("package:", PACKAGE_PATH)
print("archive:", DRIVE_ARCHIVE_PATH)
print("result:", RESULT_PATH)

package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl
archive: /content/drive/MyDrive/TorNet_Backup/tornet_2013.tar.gz
result: /content/drive/MyDrive/TorNet_Backup/experiments/io_benchmark_v1/2013_file_tensor_benchmark.json


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.7-py3-none-any.whl'], returncode=0)

In [4]:
import numpy as np

import tornado_detection
from tornado_detection.data import (
    load_canonical_frame_index,
    read_netcdf_file,
    read_netcdf_frame,
)

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

canonical_index = load_canonical_frame_index(
    MANIFESTS_ROOT
)

eligible = canonical_index.loc[
    canonical_index["year"].eq(2013)
    & canonical_index["split"].eq("train")
].copy()

member_names = (
    eligible["archive_member"]
    .drop_duplicates()
    .sort_values()
    .head(BENCHMARK_FILE_COUNT)
    .tolist()
)

if len(member_names) != BENCHMARK_FILE_COUNT:
    raise AssertionError(
        f"Expected {BENCHMARK_FILE_COUNT} files; "
        f"found {len(member_names)}"
    )

benchmark_index = (
    eligible.loc[
        eligible["archive_member"].isin(
            member_names
        )
    ]
    .sort_values(
        ["archive_member", "frame_index"]
    )
    .reset_index(drop=True)
)

frame_counts = benchmark_index.groupby(
    "archive_member"
).size()

if not frame_counts.eq(4).all():
    raise AssertionError(
        "Every benchmark file must contain four frames"
    )

expected_frame_count = BENCHMARK_FILE_COUNT * 4

if len(benchmark_index) != expected_frame_count:
    raise AssertionError(
        f"Expected {expected_frame_count} frames; "
        f"found {len(benchmark_index)}"
    )

print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print("benchmark files:", len(member_names))
print("benchmark frames:", len(benchmark_index))
print(
    "positive frames:",
    int(benchmark_index["frame_label"].sum()),
)

tornado_detection: 0.1.7
benchmark files: 512
benchmark frames: 2048
positive frames: 0


In [5]:
import shutil
import tarfile
import time

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    DRIVE_ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter() - copy_started
)

required_members = set(member_names)
extracted_members = set()
extraction_started = time.perf_counter()

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name not in required_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(member)

        if source_file is None:
            raise RuntimeError(
                f"Could not extract {member.name}"
            )

        with (
            source_file,
            destination.open("wb") as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(member.name)

extraction_seconds = (
    time.perf_counter() - extraction_started
)
missing_members = (
    required_members - extracted_members
)

if missing_members:
    raise RuntimeError(
        "Missing benchmark members: "
        f"{sorted(missing_members)[:10]}"
    )

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "extracted files:",
    len(extracted_members),
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)

copy seconds: 11.161
extracted files: 512
extraction seconds: 9.446


In [6]:
frame_read_seconds = 0.0
file_read_seconds = 0.0
verified_files = 0

for member_number, member_name in enumerate(
    member_names
):
    path = EXTRACTION_ROOT / member_name

    def read_frames_separately():
        started = time.perf_counter()

        results = [
            read_netcdf_frame(
                path,
                frame_index,
            )
            for frame_index in range(4)
        ]

        elapsed = (
            time.perf_counter() - started
        )

        values = np.stack(
            [
                result.values
                for result in results
            ],
            axis=0,
        )
        labels = np.asarray(
            [
                result.label
                for result in results
            ],
            dtype=np.uint8,
        )

        return values, labels, elapsed

    def read_whole_file():
        started = time.perf_counter()
        result = read_netcdf_file(path)
        elapsed = (
            time.perf_counter() - started
        )

        return (
            result.values,
            result.labels,
            elapsed,
        )

    if member_number % 2 == 0:
        (
            file_values,
            file_labels,
            file_elapsed,
        ) = read_whole_file()
        (
            frame_values,
            frame_labels,
            frame_elapsed,
        ) = read_frames_separately()
    else:
        (
            frame_values,
            frame_labels,
            frame_elapsed,
        ) = read_frames_separately()
        (
            file_values,
            file_labels,
            file_elapsed,
        ) = read_whole_file()

    np.testing.assert_array_equal(
        file_labels,
        frame_labels,
    )
    np.testing.assert_allclose(
        file_values,
        frame_values,
        rtol=0.0,
        atol=0.0,
        equal_nan=True,
    )

    expected_labels = (
        benchmark_index.loc[
            benchmark_index[
                "archive_member"
            ].eq(member_name),
            "frame_label",
        ]
        .astype(np.uint8)
        .to_numpy()
    )

    np.testing.assert_array_equal(
        file_labels,
        expected_labels,
    )

    frame_read_seconds += frame_elapsed
    file_read_seconds += file_elapsed
    verified_files += 1

print("verified files:", verified_files)
print(
    "separate-frame seconds:",
    round(frame_read_seconds, 3),
)
print(
    "whole-file seconds:",
    round(file_read_seconds, 3),
)

verified files: 512
separate-frame seconds: 38.461
whole-file seconds: 15.186


In [7]:
import datetime
import json

benchmark_frames = BENCHMARK_FILE_COUNT * 4

frame_reader_fps = (
    benchmark_frames / frame_read_seconds
)
file_reader_fps = (
    benchmark_frames / file_read_seconds
)
speedup = (
    frame_read_seconds / file_read_seconds
)

canonical_train_frames = 547_672

projected_frame_reader_seconds = (
    canonical_train_frames / frame_reader_fps
)
projected_file_reader_seconds = (
    canonical_train_frames / file_reader_fps
)

metrics = {
    "artifact_kind": (
        "file_tensor_io_benchmark"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": PACKAGE_VERSION,
    "year": 2013,
    "official_split": "train",
    "benchmark_file_count": (
        BENCHMARK_FILE_COUNT
    ),
    "benchmark_frame_count": (
        benchmark_frames
    ),
    "call_order": "alternating_by_file",
    "tensor_equality_verified": True,
    "label_equality_verified": True,
    "separate_frame_read_seconds": (
        frame_read_seconds
    ),
    "whole_file_read_seconds": (
        file_read_seconds
    ),
    "separate_frame_reader_fps": (
        frame_reader_fps
    ),
    "whole_file_reader_fps": (
        file_reader_fps
    ),
    "whole_file_speedup": speedup,
    "canonical_train_frame_count": (
        canonical_train_frames
    ),
    "projected_separate_frame_epoch_seconds": (
        projected_frame_reader_seconds
    ),
    "projected_whole_file_epoch_seconds": (
        projected_file_reader_seconds
    ),
    "copy_seconds": copy_seconds,
    "extraction_seconds": extraction_seconds,
}

RESULT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
)
print("wrote:", RESULT_PATH)

{
  "artifact_kind": "file_tensor_io_benchmark",
  "benchmark_file_count": 512,
  "benchmark_frame_count": 2048,
  "call_order": "alternating_by_file",
  "canonical_train_frame_count": 547672,
  "copy_seconds": 11.160987782999655,
  "created_at_utc": "2026-09-13T22:42:13.849641+00:00",
  "extraction_seconds": 9.445624824000333,
  "label_equality_verified": true,
  "official_split": "train",
  "package_version": "0.1.7",
  "projected_separate_frame_epoch_seconds": 10285.17168885568,
  "projected_whole_file_epoch_seconds": 4060.944859941024,
  "separate_frame_read_seconds": 38.461034376006864,
  "separate_frame_reader_fps": 53.24869788935274,
  "tensor_equality_verified": true,
  "whole_file_read_seconds": 15.185759128016798,
  "whole_file_reader_fps": 134.863195361868,
  "whole_file_speedup": 2.5327040980815116,
  "year": 2013
}
wrote: /content/drive/MyDrive/TorNet_Backup/experiments/io_benchmark_v1/2013_file_tensor_benchmark.json


In [8]:
shutil.rmtree(EXTRACTION_ROOT)
LOCAL_ARCHIVE_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()
assert RESULT_PATH.is_file()

print(
    "Removed all Colab-local benchmark artifacts"
)
print("Preserved:", RESULT_PATH)

Removed all Colab-local benchmark artifacts
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/io_benchmark_v1/2013_file_tensor_benchmark.json
